In [1]:
import torch
print(torch.cuda.is_available())

import sys
import torch
print("Chemin Python :", sys.executable)
print("Version PyTorch :", torch.__version__)

/home/luciacev/anaconda3/envs/paul_agent/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


True
Chemin Python : /home/luciacev/anaconda3/envs/paul_agent/bin/python
Version PyTorch : 2.6.0+cu124


# Imports and Initial Configuration

In [2]:
import json
import time
import numpy as np
from numpy.linalg import norm
from pydantic import BaseModel, Field
from ollama import chat, embeddings
from pathlib import Path
from sentence_transformers import CrossEncoder
import torch

# ----- GLOBAL VARIABLES -----
model = "llama3:latest"
# model = "qwen3:8b"
queries_type = "new_queries" 
top_k = 3 # Number of top tools to retrieve

# ----- GLOBAL VARIABLES -----
script_folder = Path().absolute()
tools_path = script_folder.parent / 'input' / 'documentation' / 'tools'
queries_path = script_folder.parent / 'input' / 'param' / f'{queries_type}.json'
result_path = script_folder / 'output' / f'all_{queries_type}.json'

/home/luciacev/anaconda3/envs/paul_agent/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch

# RAG Retrieval Logic

In [4]:
# ----- Cross Encoder (Reranker) -----
def reranker(tools, user_prompt, top_k):
    pairs = []
    for tool in tools:
        formatted_text = json.dumps(tool)
        pairs.append([user_prompt, formatted_text])
    
    scores = reranker_model.predict(pairs)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for i in top_indices:
        results.append({"score": float(scores[i]), "tool": tools[i]})
    return results

# ----- Router LLM-----
class RouteDecision(BaseModel):
    reasoning: str = Field(description="step-by-step analysis comparing the user's request against the available tools before making a decision.")
    selected_tool: str = Field(description="The exact name of the tool. Return 'none' if no tool matches.")

def router(relevant_tools, model_name, user_prompt):
    tools_formatted = "\n".join([
        f"{t.get('name', 'Unknown')}: {t.get('description', '')} (Tags: {', '.join(t.get('tags', []))})" 
        for t in relevant_tools
    ])
    
    system_prompt = f"""You are a Router Agent expert in medical and dental imaging (CBCT, IOS, MRI).
    Your role is to analyze the user's request and select the most relevant tool from the FILTERED list below.
    If none of these {len(relevant_tools)} tools fit perfectly, return 'none'.

    === FILTERED TOOLS ===
    {tools_formatted}

    === ROUTING GUIDELINES & EXAMPLES ===
    - Pay close attention to subtle differences. For example, if a user specifically asks for "batch processing" or "multiple scans", prioritize tools designed for batching (e.g., batchdentalseg).
    - If a user asks to "segment" or "split" specific teeth, ensure the tool handles instance segmentation (e.g., amasss_cli).
    - If the request is for registration, check if it's CBCT-to-CBCT, MRI-to-CBCT, or intraoral (IOS) and choose the specific tool accordingly.

    Carefully analyze the user's prompt step-by-step in the 'reasoning' field BEFORE selecting the tool. Output strictly matching the JSON schema.
    """
    try: 
        response = chat(
            model=model_name,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            format=RouteDecision.model_json_schema(),
            options={"temperature":0},

        )
        return RouteDecision.model_validate_json(response.message.content)
    except Exception as e:
        return RouteDecision(selected_tool="error", reasoning=f"Error : {str(e)}")

# Agent Logic

In [5]:
class ParameterExtraction(BaseModel):
    reasoning: str = Field(description="Step-by-step reasoning for extraction these parameters from user's prompt.")
    parameters: dict = Field(description="Extracted parameters as a key-value dictionary. Use the exact parameter names from the tool schema.")

def agent(selected_tool_dict, model_name, user_prompt):
    tool_formatted = json.dumps(selected_tool_dict)

    system_prompt = f"""You are en expert Parameter Extraction Agent for medical and dental imaging tools.
    Your task is to analyse the user's request and extract the required parameters according to the selected tool's schema
    
    SELECTED TOOL SCHEMA:
    {json.dumps(tool_formatted, indent=2)}

    carefully read the user's prompt and extract the appropriate values for the tool's parameters.
    Output strictly matching the JSON schema. if an optional parameter is missing, do not invent it"""

    try:
        response = chat(
            model = model_name,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            format=ParameterExtraction.model_json_schema(),
            options={"temperature": 0.0, "num_predict": 250},
        )
        return ParameterExtraction.model_validate_json(response.message.content)
    except Exception as e:
        return ParameterExtraction(parameters={}, reasoning=f"Error: {str(e)}")


# Terminal Output Display Functions

In [6]:
def evaluate_reranker(reranker_result,expected_tool,index,total_queries,prompt):

    reranker_tools = [res["tool"].get("name") for res in reranker_result]
    reranker_match = expected_tool in reranker_tools
    reranker_perfect = (expected_tool == reranker_tools[0])

    if reranker_perfect:
        raranker_sign = "✅"
    elif reranker_match:
        raranker_sign = "☑️"
    else: 
        raranker_sign = "❌"
    
    print(f"\n{'='*65}")
    print(f"Prompt ({index}/{total_queries} : '{prompt}'")
    print(f"{'-'*65}")
    print(f"I Reranker : {raranker_sign}")
    print(f"{'-'*65}")

    for i,res in enumerate(reranker_result,start=1):
        tool_name = res["tool"].get('name', 'Unknow')
        if tool_name == expected_tool:
            is_expected = "->"
        else:
            is_expected = "  "
        
        print(f"{is_expected} {i} Score: {res['score']:.4f} tool : {tool_name}")
    print(f"{'-'*65}")
    return reranker_perfect, reranker_match, reranker_tools


def evaluate_router(decision, expected_tool):
    print(f"II Router")
    print(f"{'-'*65}")
    
    router_match = (decision.selected_tool == expected_tool)
    if router_match:
        print(f"Tool chosen : {decision.selected_tool} ✅")
    else:
        print(f"Tool chosen : {decision.selected_tool}")
        print(f"Expected: {expected_tool} ❌")
    print(f"{'-'*65}")
    return router_match

def evaluate_agent(extracted_params, expected_params):
    print(f"III Agent")
    print(f"{'-'*65}")

    agent_correct = 0
    # expected param
    for key, expected_val in expected_params.items():
        extracted_val = extracted_params.get(key)
        if extracted_val == expected_val:
            print(f' "{key}": {json.dumps(expected_val)} ✅')
            agent_correct +=1
        else:
            print(f' "{key}": {json.dumps(expected_val)} ❌ (Extract:{json.dumps(extracted_val)})')
        
    # hallucination
    for key, extracted_val in extracted_params.items():
        if key not in expected_params:
            print(f' "{key}": {json.dumps(extracted_val)} ⚠️ (Hallucination)')
    print(f"{'-'*65}")

# Calcul de la perfection (0 ou 1)
    if len(expected_params) > 0:
        agent_perfect = (agent_correct == len(expected_params)) and (len(extracted_params) == len(expected_params))
    else:
        agent_perfect = (len(extracted_params) == 0)

    return agent_perfect, agent_correct


    


# Main Loop

In [7]:
reranker_correct_count = 0
reranker_perfect_count = 0
router_perfect_count = 0
agent_correct_count = 0
agent_perfect_count = 0
total_expected_params = 0
results_detail = []
decision = None
reranker_model = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda")

# ----- Load Files -----
tools = []
for file_path in tools_path.glob('*.json'):
    with open(file_path, 'r', encoding='utf-8') as f:
        tools.append(json.load(f))

with open(queries_path, 'r', encoding='utf-8') as f:
    queries = json.load(f)

total_queries = int(len(queries))


# ----- Loop-----
for index,q in enumerate(queries, start=1):
    prompt = q['query']
    expected_tool = q['expected_tool']
    expected_params = q.get('expected_params',{})
    total_expected_params += len(expected_params)

    # I Reranker
    reranker_result = reranker(tools, prompt, top_k) # Run reranker
    reranker_perfect, reranker_correct, reranker_tools = evaluate_reranker(reranker_result,expected_tool,index,total_queries,prompt)

    if reranker_correct: reranker_correct_count +=1
    if reranker_perfect: reranker_perfect_count +=1

    # II Router 
    top_tools_dicts = [res["tool"] for res in reranker_result]
    decision = router(top_tools_dicts, model, prompt)
    selected_tool = decision.selected_tool
    router_perfect = evaluate_router(decision, expected_tool)

    if router_perfect: router_perfect_count +=1

    # III Agent 
    extracted_params = {}
    if router_perfect:
        for tool in tools:
            if tool.get("name") == decision.selected_tool:
                    selected_tool_dict = tool
                    break
            
        if selected_tool_dict:
            param_response = agent(selected_tool_dict, model, prompt)
            extracted_params = param_response.parameters

            agent_perfect, agent_correct = evaluate_agent(extracted_params, expected_params)

            agent_correct_count += agent_correct
            if agent_perfect: 
                agent_perfect_count += 1

    results_detail.append({
        "prompt": prompt,
        "expected_tool": expected_tool,
        "selected_tool": selected_tool,
        "expected_params": expected_params,
        "selected_params": extracted_params,

    })
    


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3614.91it/s]



Prompt (1/230 : 'Find landmarks 1,2 and 3 on the CBCT scans in /data/patients/cbct_batch1. Use the ALI models from /models/ali_cbct and save the results to /output/landmarks. Use /tmp/ali_work as scratch space. These are regular NIfTI files, not DICOM.'
-----------------------------------------------------------------
I Reranker : ✅
-----------------------------------------------------------------
-> 1 Score: 0.7884 tool : ali_cbct
   2 Score: 0.5741 tool : semi_aso_cbct
   3 Score: 0.2996 tool : areg_cbct
-----------------------------------------------------------------
II Router
-----------------------------------------------------------------
Tool chosen : ali_cbct ✅
-----------------------------------------------------------------
III Agent
-----------------------------------------------------------------
 "input": "/data/patients/cbct_batch1" ✅
 "dir_models": "/models/ali_cbct" ✅
 "lm_type": "1,2,3" ✅
 "output_dir": "/output/landmarks" ✅
 "temp_fold": "/tmp/ali_work" ✅
 "DCMInput

In [8]:
agent_correct_percentage = (agent_correct_count / total_expected_params * 100) if total_expected_params > 0 else "Na"
reranker_correct_percentage = (reranker_correct_count / total_queries * 100) if total_queries > 0 else "Na"
reranker_perfect_percentage = (reranker_perfect_count / total_queries * 100) if total_queries > 0 else "Na"
router_perfect_percentage = (router_perfect_count / total_queries * 100) if total_queries > 0 else "Na"
agent_perfect_percentage = (agent_perfect_count / total_queries * 100) if total_queries > 0 else "Na"

agent_correct = (agent_correct)
summary = {
    "queries_type": queries_type,
    "model": model,
    "metrics": {
        "total_queries": total_queries,
        "Reranker_correct": reranker_correct_count,
        "Reranker_perfect": reranker_perfect_count,
        "Router_accuracy": router_perfect_count,
        "Agent_Param Correct": round(agent_correct_percentage,2),
        "Agent_Param Perfect": agent_perfect_count,
    },
    "details": results_detail,
}

print(f"Total quereies: {total_queries}")
print(f"Reranker (top-3):     {round(reranker_correct_percentage,2)}% ({reranker_correct_count}/{total_queries})")
print(f"Reranker (top-1):     {round(reranker_perfect_percentage,2)}% ({reranker_perfect_count}/{total_queries})")
print(f"Router (Exact):       {round(router_perfect_percentage,2)}% ({router_perfect_count}/{total_queries})")
print(f"Agent Param Correct:  {round(agent_correct_percentage,2)}% ({agent_correct_count}/{total_expected_params})")
print(f"Agent Param Perfect:  {round(agent_perfect_percentage,2)}% ({agent_perfect_count}/{total_queries})")

result_path.parent.mkdir(parents=True, exist_ok=True)
with open(result_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

print(f"\n Save: {result_path.resolve()}")

Total quereies: 230
Reranker (top-3):     99.57% (229/230)
Reranker (top-1):     94.78% (218/230)
Router (Exact):       94.35% (217/230)
Agent Param Correct:  85.33% (983/1152)
Agent Param Perfect:  59.13% (136/230)

 Save: /media/luciacev/Data/Paul_Agent/ALL/output/all_new_queries.json
